# 07 — Evaluation (Kaggle eval kernel)

Run the held-out competition benchmark, leakage checks, and base-capability regression probes on a Kaggle GPU, then write `output/evaluation-results.json` — the file the promotion gate (`scripts/verify_release.py`, via `scripts/evaluate_and_promote.py`) reads on the GitHub runner.

Mode comes from `eval-mode.json` (written by `scripts/package_eval_kernel.py`): `candidate` evaluates the trained adapter mounted from the training kernel (`/kaggle/input/`, resolved by searching for `adapter_config.json`); `base` evaluates the pinned base model, which is how the baseline gets measured. Scores are never invented: without a solver attached, competitions are recorded `not_run` with null scores, the gate fails closed, and the candidate stays private.

This notebook never publishes. Publication happens only on the GitHub runner, after the gate passes.

In [ ]:
import sys
sys.path.insert(0, ".")
from pathlib import Path
print("repo root:", Path.cwd())
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_GPU = False
print("GPU available:", HAS_GPU)
if not HAS_GPU:
    print("Skipping GPU steps on this host — run this notebook on Kaggle with a GPU.")

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

def _find_adapter_dir() -> Path | None:
    """Locate the trained adapter: Kaggle kernel input first."""
    env_dir = os.environ.get("ADAPTER_DIR")
    if env_dir and Path(env_dir).exists():
        return Path(env_dir)
    for candidate in Path("/kaggle/input").glob("**/adapter_config.json"):
        return candidate.parent
    local = Path("output/akarui-sft-v0")
    return local if local.exists() else None

mode = "candidate"
mode_file = Path("eval-mode.json")
if mode_file.exists():
    mode = json.loads(mode_file.read_text()).get("mode", mode)
mode = os.environ.get("EVAL_MODE", mode)
print("eval mode:", mode)

if HAS_GPU:
    if mode == "base":
        model_arg = "base"
    else:
        adapter_dir = _find_adapter_dir()
        if adapter_dir is None:
            raise RuntimeError(
                "candidate mode but no adapter found: expected the "
                "training kernel output under /kaggle/input/, "
                "$ADAPTER_DIR, or output/akarui-sft-v0"
            )
        model_arg = str(adapter_dir)
    print("evaluating model:", model_arg)
    result = subprocess.run(
        [sys.executable, "scripts/evaluate.py",
         "--config", "configs/evaluation.yaml",
         "--model", model_arg,
         "--output", "output/evaluation-results.json"],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"evaluation failed (exit {result.returncode})")
    results = json.loads(Path("output/evaluation-results.json").read_text())
    print("model:", results["model"])
    print("normalized score:", results["normalized_score"])
    print("submission validity:", results["submission_validity"])
    for entry in results["results"]:
        print("-", entry["competition_id"], entry["status"])
else:
    print("No GPU: evaluation skipped.")

In [ ]:
from src.evaluation.leakage_detection import detect_leakage
import json
manifest = json.load(open("evaluation/benchmark_manifest.json"))
held_out = [c["id"] for c in manifest["held_out_competitions"]]
print("held-out competitions:", len(held_out))
train_file = Path("data/train.jsonl")
if train_file.exists():
    train_ids = [json.loads(line).get("competition_id", "") for line in train_file.read_text().splitlines() if line.strip()]
    report = detect_leakage(train_ids, held_out)
    print("leakage report:", report)
else:
    print("No training file in this kernel; leakage was checked at packaging time.")

In [ ]:
from src.evaluation.regression_tests import run_regression
report = run_regression(None, "evaluation/regression_probes.json", "evaluation/baseline_results.json", "output/regression.json")
print("probes total:", report["total"], "| not run (no model attached):", report["not_run"])
print("regressions:", report["regressions"])